## I am trying to implement a basic Agent with Rag on my Custom DataSet which contains some txt file where my data is stored

In [82]:
from dotenv import load_dotenv

load_dotenv()

True

In [83]:
from llama_index.core.agent import FunctionAgent
from llama_index.llms.ollama import Ollama
from llama_index.core.workflow import context
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.core import Settings
import pprint


In [84]:
Settings.llm = Ollama(model="qwen2.5:3b", request_timeout=360.0)
Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text")

# 2. Build the index from your local dataset folder
documents = SimpleDirectoryReader("../Dataset").load_data()
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()

In [85]:
async def get_info_from_dataset(query: str) -> str:
    """Queries the internal document database/dataset to find specific, factual information 
    regarding the context files. Use this whenever the user asks about indexed data.
    
    Args:
        query (str): The search query to locate in the database.
    """
    try:
        response = await query_engine.aquery(query)
        return str(response)
    except Exception as e:
        return f"Error retrieving information: {e}"

In [86]:
from llama_index.core.tools import FunctionTool

dataset_tool = FunctionTool.from_defaults(async_fn=get_info_from_dataset)

In [ ]:
llm = Ollama(model="qwen2.5:3b", request_timeout=120.0)

agent = FunctionAgent(
    llm= llm,
    tools=[get_info_from_dataset],
    system_prompt=(
   """You are a helpful assistant with access to an internal document database containing 
indexed files (text and PDF documents) from the user's Dataset folder.

You have access to the following tool:

- get_info_from_dataset(query: str): Queries the internal document database to find 
  specific, factual information from the indexed context files. Use this tool whenever 
  the user asks a question that could be answered by information contained in the 
  indexed documents — including facts, definitions, figures, names, dates, procedures, 
  or any domain-specific content that may not be part of your general knowledge.

Guidelines:
1. ALWAYS call get_info_from_dataset first when the user's question relates to specific 
   documents, internal data, company information, or anything that sounds like it could 
   be sourced from a private/local dataset rather than general world knowledge.
2. Do NOT rely on your own prior/general knowledge to answer questions that are likely 
   covered by the dataset. Prefer the tool's retrieved context over assumptions.
3. If the tool returns no relevant information or an empty/unhelpful response, tell the 
   user clearly that the information wasn't found in the indexed documents, rather than 
   guessing or fabricating an answer.
4. When you do use retrieved information, synthesize it into a clear, concise, natural 
   language answer — don't just dump raw retrieved text unless the user asks for exact 
   quotes.
5. If the user's query is ambiguous, ask a clarifying question before querying the 
   dataset, so the query passed to get_info_from_dataset is specific and well-formed.
6. If the query clearly does NOT relate to the dataset (e.g. casual conversation, math, 
   general reasoning), respond directly without calling the tool.
7. Cite which topic/section of the dataset the information seems to come from, if that 
   context is available in the tool's response, so the user can trust the source.

Your goal is to be an accurate, grounded assistant that prioritizes retrieved factual 
context from the internal dataset over speculation."""
),
verbose=True,
)

In [92]:
response = await agent.run("Summarize PRODUCT FAQ - SmartHome Hub X200")
print(response)

The PRODUCT FAQ - SmartHome Hub X200 provides information about its battery backup duration during power outages which is up to 6 hours. It can connect up to 150 smart devices including those using Zigbee, Z-Wave, and Wi-Fi protocols. The warranty for the product includes a standard two-year coverage against manufacturing defects, with an option available for an extended five-year warranty. The Pro subscription, priced at $4.99 per month, offers advanced features like AI-based anomaly detection and cloud video storage. Upon its launch, the SmartHome Hub X200 is priced at $129.99 USD, and it is compatible with voice assistants such as Amazon Alexa, Google Assistant, and Apple HomeKit.


In [93]:
response = await agent.run("Summarize CUSTOMER SUPPORT COMMUNICATION TRANSCRIPT: E-COMMERCE DOMAIN")
print(response)

The CUSTOMER SUPPORT COMMUNICATION TRANSCRIPT involved case ID EC-994821-X and occurred on July 20, 2026. The main issue was a damaged item (Lumos Ceramic Table Lamp) where the first-time customer discount code WELCOME15 wasn't applied correctly during checkout. Customer David K. contacted OmniShop Support through live chat and email with a Senior Support Specialist named Sarah M., who resolved the issue by initiating an expedited replacement order for the broken lamp, to be shipped within 2 business days at no extra cost to the customer. Additionally, a refund of $18.00 was provided for the discount that wasn't applied due to system limitations.
